# Importação das Bibliotecas

In [1]:
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
import plotly.express as px
import matplotlib.pyplot as plt
import spacy as sp
import re

# Limpeza e pré-processamento de dados

garanta que os dados estejam limpos, consistentes e bem formatados. Lidar com valores ausentes, padronizar formatos de data e garantir codificação de texto consistente tornará o conjunto de dados mais amigável ao usuário.

In [2]:
data = pd.read_csv("/content/fashionnova_reviews.csv")
data.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Champagne,/users/66c78240087b6269ffbcb5fb,US,1 review,2024-08-22T20:24:02.000Z,Rated 5 out of 5 stars,I love ordering from fashion nova,I love ordering from fashion nova. The clothes...,"August 22, 2024"
1,Vg customer,/users/6618fdb53d4198001210cbe7,VG,3 reviews,2024-08-21T05:43:11.000Z,Rated 5 out of 5 stars,Top tier content for fashion nova,Always amazing clothes and the fast shipping i...,"August 18, 2024"
2,Colleen Burgher,/users/64e9595206be1a001244ff73,US,3 reviews,2024-08-21T17:09:14.000Z,Rated 5 out of 5 stars,Prices and quality of products are…,Prices and quality of products are GREAT Would...,"August 21, 2024"
3,R.G.M,/users/66c58ad1c6ab36352a08f57a,US,1 review,2024-08-21T08:36:03.000Z,Rated 5 out of 5 stars,Great customer service,Great customer service. I was helped until the...,"August 20, 2024"
4,Rosalyn Cousar,/users/60ad4b6ef3788e001adbb8e3,US,5 reviews,2024-08-22T00:46:16.000Z,Rated 3 out of 5 stars,False advertising,Disappointing experience. You don’t live up to...,"August 21, 2024"


In [3]:
data = data.sample(n=20000, random_state=42).reset_index(drop=True)

In [4]:
data.duplicated().sum()

3

In [5]:
data.isnull().sum()

,0
Reviewer Name,12
Profile Link,0
Country,0
Review Count,0
Review Date,0
Rating,0
Review Title,5
Review Text,0
Date of Experience,0


In [6]:
data.dropna(inplace=True, axis=0)
data.drop_duplicates(inplace=True)

In [7]:
def processing_rating(rating):
  rating = rating.split(" ")[1]
  rating = int(rating)
  return rating

In [8]:
data["Rating"] = data["Rating"].apply(processing_rating)

In [9]:
# data["Date of Experience"] =  pd.to_datetime(data["Date of Experience"]).dt.year

# Análise de atendimento ao cliente

Entenda a qualidade do atendimento ao cliente e as áreas que precisam de melhorias.

In [ ]:
average_ratings = data.groupby("Country")["Rating"].mean()
average_ratings = average_ratings.reset_index()
average_ratings['Rating'] = average_ratings['Rating'].round(3)

fig = px.scatter(average_ratings, x=average_ratings['Country'],
                 y=average_ratings['Rating'], size="Rating", color="Country")

fig.update_layout(xaxis_title=None, yaxis_title=None)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.show()

# Análise de sentimento de dados

Ao analisar o texto e o título da avaliação,
você pode realizar uma análise de
sentimento para entender o sentimento
geral do cliente em relação à Fashion Nova.

In [11]:
for _ in range(10):
  print(data["Review Text"][np.random.randint(0, len(data["Review Text"]))])

Review text not found
I love it very professionalism clothes are beautiful
Fashion Nova clothing is either hit or miss! Sometimes the clothing quality is good and sometimes it’s bad! One time I ordered a sweater material onesie and it came with the whole arm ripped. I tried to call or emailCustomer service but of course they make it so you can’t get in contact with anyone. The prices are also always fluctuating kinda like they price gauge and then they will offer you all these coupons and act like you are saving so much! One day I can order $300 worth of stuff and the very next day the items are $200…now can I call or email anyone to request a price adjustment??? No I have to live w/the mistake of not waiting until the next day! Some days are a hit others are a miss! U get what u pay for! I do always look good in most of the things I buy though and everything is typically true to size!
I love the super cute clothes at the super low prices!!!
Paid 201 for dress. Was Buy 1 get one Free P

In [12]:
data["Review Text"] = data["Review Text"].str.replace("Review text not found", "")
data = data[data["Review Text"] != ""]

In [13]:
nlp = sp.load("en_core_web_sm")

In [14]:
def clean_text(text):
  text = text.lower()
  text = text.replace("\n", "")
  text = re.sub(r'[^\w\s]', '', text)
  doc = nlp(text)
  tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct]
  text = " ".join(tokens)
  return text

In [15]:
reviws_text = [clean_text(text) for text in data["Review Text"]]
ratings = data["Rating"].values

In [16]:
training_sentences, testing_sentences, training_labels, testing_labels = train_test_split(reviws_text, ratings, test_size=0.2, stratify=ratings)

In [17]:
max_len = max(len(sentence.split()) for sentence in reviws_text)
print("Maior sequência de palavras:", max_len)

Maior sequência de palavras: 800


In [18]:
num_classes = len(np.unique(ratings))
embedding_dim = 100
num_epochs = 15
vocab_size = 1000
oov_token = "<OOV>"

In [19]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(training_sentences)

training_sequences = tokenizer.texts_to_sequences(training_sentences)
training_padded = pad_sequences(training_sequences, maxlen=max_len, padding='post', truncating='post')

testing_sequences = tokenizer.texts_to_sequences(testing_sentences)
testing_padded = pad_sequences(testing_sequences, maxlen=max_len, padding='post', truncating='post')

training_labels = training_labels - 1
testing_labels = testing_labels - 1

In [20]:
class ClassifierModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, num_classes):
    super(ClassifierModel, self).__init__()
    self.embedding = Embedding(vocab_size, embedding_dim)
    self.lstm1 = Bidirectional(LSTM(100, return_sequences=True))
    self.lstm2 = Bidirectional(LSTM(100, return_sequences=True))
    self.lstm3 = Bidirectional(LSTM(100))
    self.dense1 = Dense(64, activation='relu')
    self.dense2 = Dense(num_classes)

  def call(self, inputs):
    x = self.embedding(inputs)
    x = self.lstm1(x)
    x = self.lstm2(x)
    x = self.lstm3(x)
    x = self.dense1(x)
    x = self.dense2(x)
    return x

In [21]:
classifier = ClassifierModel(vocab_size, embedding_dim, num_classes)
classifier.compile(loss=SparseCategoricalCrossentropy(from_logits=True),
                optimizer=Adam(learning_rate=0.001),
                metrics=['accuracy'])

In [ ]:
early_stopping = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history = classifier.fit(training_padded, training_labels, epochs=num_epochs,
                         validation_data=(testing_padded, testing_labels),
                         batch_size=32, callbacks=[early_stopping])

Epoch 1/15
359/359 ━━━━━━━━━━━━━━━━━━━━ 75s 186ms/step - accuracy: 0.6804 - loss: 0.9490 - val_accuracy: 0.7223 - val_loss: 0.7819
Epoch 2/15
359/359 ━━━━━━━━━━━━━━━━━━━━ 79s 185ms/step - accuracy: 0.7277 - loss: 0.7620 - val_accuracy: 0.7195 - val_loss: 0.7527
Epoch 3/15
359/359 ━━━━━━━━━━━━━━━━━━━━ 82s 185ms/step - accuracy: 0.7396 - loss: 0.7124 - val_accuracy: 0.7341 - val_loss: 0.7270
Epoch 4/15
359/359 ━━━━━━━━━━━━━━━━━━━━ 82s 185ms/step - accuracy: 0.7560 - loss: 0.6745 - val_accuracy: 0.7334 - val_loss: 0.7486
Epoch 5/15
359/359 ━━━━━━━━━━━━━━━━━━━━ 82s 185ms/step - accuracy: 0.7594 - loss: 0.6671 - val_accuracy: 0.7331 - val_loss: 0.7442
Epoch 6/15
359/359 ━━━━━━━━━━━━━━━━━━━━ 86s 197ms/step - accuracy: 0.7660 - loss: 0.6476 - val_accuracy: 0.7282 - val_loss: 0.7461


In [ ]:
def plot_training_history(history):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Treino')
    plt.plot(history.history['val_loss'], label='Validação')
    plt.title('Perda')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Treino')
    plt.plot(history.history['val_accuracy'], label='Validação')
    plt.title('Acurácia')
    plt.legend()

    plt.show()

In [ ]:
plot_training_history(history)

# Análise temporal

com a Data de revisão e a Data da experiência,
você pode analisar como o feedback do cliente varia ao longo do tempo,
o que é útil para entender o impacto de eventos ou promoções específicas.